# 08a · Prepare the data for GPU training on Colab: BankSight

**Input given to the transformer:** `product | issue | narrative`. Putting product and issue in front of the text gives the transformer the same
information B4 has, so the comparison is fair.

**What this does:** loads the libraries.

In [1]:
from pathlib import Path

import duckdb
import joblib
import pandas as pd

pd.set_option("display.width", 170)

**What this does:** sets the file locations. The export goes to `data/colab/`.

In [2]:
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
WAREHOUSE = ROOT / "data" / "warehouse" / "banksight.duckdb"
EXPORT_DIR = ROOT / "data" / "colab"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
PII_FILE = ROOT / "models" / "07_pii_patterns.joblib"

assert WAREHOUSE.exists(), "Run notebooks 01 and 01b first."
assert PII_FILE.exists(), "Run notebook 07 first (it saves the redaction patterns)."

MAX_CHARS = 1800   # a transformer reads about 256 tokens (roughly 1,300 characters), so longer text is cut off anyway; this keeps the file small

**What this does:** reads train and validation complaints with a final outcome. The test period is excluded in the SQL itself.

In [3]:
con = duckdb.connect(str(WAREHOUSE), read_only=True)
df = con.sql("""
    SELECT l.complaint_id, l.date_received, l.product_group, r.issue, l.got_relief, l.split, r.narrative
    FROM features.complaint_labels l
    JOIN raw.cfpb_complaints r USING (complaint_id)
    WHERE l.target_eligible
      AND l.split IN ('train', 'validation')
      AND r.narrative IS NOT NULL AND length(trim(r.narrative)) > 0
""").df()
con.close()

df["issue"] = df["issue"].fillna("Unknown").astype(str)
df = df.sort_values("date_received", kind="stable").reset_index(drop=True)
print(df["split"].value_counts().to_string())

split
train         103777
validation     18537


**What this does:** replaces emails, phone numbers and similar patterns with tags such as `[EMAIL]`.

In [4]:
pii_patterns = joblib.load(PII_FILE)

text = df["narrative"]
for name, pattern in pii_patterns.items():
    text = text.str.replace(pattern, f"[{name.upper()}]", regex=True)

df["narrative_redacted"] = text.str.slice(0, MAX_CHARS)
print("patterns applied:", list(pii_patterns.keys()))

patterns applied: ['email', 'phone', 'ssn_like', 'long_digit_string', 'web_link']


**What this does:** builds the single text string the transformer will read: `product | issue | narrative`.

In [5]:
df["input_text"] = df["product_group"] + " | " + df["issue"] + " | " + df["narrative_redacted"]
df["label"] = df["got_relief"].astype(int)
print(df["input_text"].iloc[0][:300])

Mortgage | Trouble during payment process | Flagstar Bank To whom it may concern : Through the following letter I want to complain about the problems I have had with my mortgage collections since XX/XX/2022. The XXXX XXXX XXXX XXXX XXXX XXXX XXXX through the government of Florida  has sent checks fo


**What this does:** saves two small files, one for training and one for validation, with just the columns Colab needs.

In [6]:
columns = ["complaint_id", "date_received", "input_text", "label"]
train_export = df[df["split"] == "train"][columns]
valid_export = df[df["split"] == "validation"][columns]

train_export.to_parquet(EXPORT_DIR / "train.parquet", index=False)
valid_export.to_parquet(EXPORT_DIR / "validation.parquet", index=False)

for file_name in ("train.parquet", "validation.parquet"):
    size_mb = (EXPORT_DIR / file_name).stat().st_size / 1e6
    print(f"{file_name}: {size_mb:.1f} MB")
print(f"train rows {len(train_export):,} | validation rows {len(valid_export):,}")

train.parquet: 46.1 MB
validation.parquet: 9.3 MB
train rows 103,777 | validation rows 18,537


## What to do next
1. Open Google Drive and create a folder named **`banksight_colab`** in "My Drive".
2. Upload the two files from `data/colab/` (`train.parquet` and `validation.parquet`) into that folder.
3. Open **08b_colab_finetune_distilbert.ipynb** in Google Colab (upload it at colab.research.google.com), choose **Runtime, Change runtime type, T4 GPU**, and run it top to bottom.

## How to explain this in an interview
> "I train the heavy model on a free cloud GPU, but I only send train and validation text, with personal-data patterns redacted. The sealed test set stays local."